**Identificação do aluno**

**Email: levi.pereira.junior@ccc.ufcg.edu.br**

**Matrícula:**

# Laboratório: DistilBERT com Máscara de Linguagem (IMDB)

Diferente dos laboratórios anteriores, onde a maioria das seções já estava pronta, **neste laboratório você será o protagonista da construção**.  
Você deverá se basear no exemplo visto em aula e **desenvolver sua própria versão**, colocando sua **personalidade** e aplicando **toques extras** para buscar um melhor desempenho.  

Relembrando o laboratório exemplificado em aula:  
- Ele utiliza o modelo **DistilBERT** em uma tarefa de **Masked Language Modeling (MLM)**.  
- O objetivo é treinar o modelo em um conjunto de dados (IMDB) e observar sua capacidade de prever palavras mascaradas em frases.  
- O processo envolve:  
  1. Preparação dos dados.  
  2. Tokenização.  
  3. Criação de máscaras nas frases.  
  4. Fine-tuning do modelo.  
  5. Avaliação dos resultados.  

Agora é a sua vez!  
Use o que foi mostrado em aula como **guia**, mas explore variações. Use essas **DICAS**:  
- Experimente **diferentes taxas de aprendizado**.  
- Aplique **diferentes tamanhos de batch**.  
- Teste **número de épocas distintos**.
- Use **diferentes taxas de mascaramento**.   
- Pense em **formas criativas de avaliar o desempenho**.  

O objetivo não é apenas replicar o exemplo, mas **melhorá-lo com suas próprias ideias**.

# Bibliotecas

In [ ]:
!pip install torch
!pip install torch transformers huggingface_hub tokenizer

In [ ]:
import torch
import numpy as np
import tensorflow as tf
from datasets import DatasetDict
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer, AutoModelForSequenceClassification

# Conjunto de Dados

In [ ]:
# Esse conjunto de dados é relacionado ao IMDB (Plataforma que contempla resenhas sobre filmes)
dataset = load_dataset('imdb')
print(dataset['train'][0]['text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [ ]:
def stratified_subset_dataset(dataset_dict, n_samples_per_split, seed=42):
    """
    Recebe um DatasetDict (por exemplo, {'train': ..., 'test': ...})
    e retorna outro DatasetDict com a mesma estrutura,
    mas com no máximo n_samples_per_split exemplos em cada split,
    mantendo a proporção das labels.
    """
    new_splits = {}

    for split_name, split_data in dataset_dict.items():
        labels = np.array(split_data['label'])
        indices = np.arange(len(labels))
        # Seleção estratificada
        selected_indices, _ = train_test_split(
            indices,
            train_size=min(n_samples_per_split, len(labels)),
            stratify=labels,
            random_state=seed
        )
        new_splits[split_name] = split_data.select(selected_indices)

    return DatasetDict(new_splits)


subset_dataset = stratified_subset_dataset(dataset, n_samples_per_split=2000)  # DICA: tente modificar o parâmetro n_samples_per_split

In [ ]:
print(len(subset_dataset['train']))
print(len(subset_dataset['test']))

2000
2000


*   ## **Para as próximas etapas use o laborátorio apresentado em aula**

# Tokenização

Nessa seção vocês vão aprender a **transformar frases em tokens** (unidades menores de texto, como palavras ou subpalavras), que é o formato que o modelo consegue entender.  

Lembre-se:  
- O modelo **DistilBERT** não trabalha diretamente com texto cru, mas sim com **IDs numéricos** que representam tokens.  
- O **tokenizer** faz esse processo automaticamente:  
  1. Divide a frase em tokens.  
  2. Converte os tokens para IDs.  
  3. Garante que todos os exemplos tenham o mesmo tamanho (com *padding* e *truncation*). Lembre-se de tentar diferentes tamanhos de tokens de entrada do modelo.  

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Re-adding this line to define the padding token

print(f"Tokenizer pad_token_id: {tokenizer.pad_token_id}")
print(f"Tokenizer cls_token_id: {tokenizer.cls_token_id}")
print(f"Tokenizer sep_token_id: {tokenizer.sep_token_id}")
print(f"Tokenizer mask_token_id: {tokenizer.mask_token_id}")

Tokenizer pad_token_id: 0
Tokenizer cls_token_id: 101
Tokenizer sep_token_id: 102
Tokenizer mask_token_id: 103


In [ ]:
def tokenize_function(examples):
    # Adicione padding='max_length' para garantir que todas as sequências tenham o mesmo comprimento, se o batch_size não for uniforme
    return tokenizer(examples["text"], truncation=True, padding=True)

tokenized_dataset = subset_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset['train'][0])
print(tokenizer.decode(tokenized_dataset['train'][0]['input_ids']))

{'text': 'I bought this video at Walmart\'s $1 bin. I think I over-paid!!! In the 1940s, Bela Lugosi made a long string of 3rd-rate movies for small studios (in this case, Monogram--the ones who made most of the Bowry Boys films). While the wretchedness of most of these films does not approach the level of awfulness his last films achieved (Ed Wood "classics" such as Bride of the Monster and Plan 9 From Outer Space), they are nonetheless poor films and should be avoided by all but the most die-hard fans.<br /><br />I am an old movie junkie, so I gave this a try. Besides, a few of these lesser films were actually pretty good--just not this one.<br /><br />Lugosi is, what else, a mad scientist who wants to keep his rather bizarre and violent wife alive through a serum he concocts from young brides. They never really explained WHY it had to be brides or why it must be women or even what disease his wife had--so you can see that the plot was never really hashed out at all.<br /><br />Anywa

# Adicionando Mascara aos Dados

Nessa etapa vocês vão aplicar a **máscara de linguagem (MLM)**, que é o coração do pré-treinamento do BERT/DistilBERT.  
A ideia é **esconder (mascarar) aleatoriamente tokens** da frase e pedir para o modelo tentar prever quais palavras estavam lá.  

**Dica:**
- Não é preciso reinventar tudo: a biblioteca `transformers` já possui funções que ajudam a criar essas máscaras.  
- Testem mascarar as frases considerando outras proporções além dos 15% originais do artigo.   

In [ ]:
def mask_tokens_manual(examples, tokenizer, mlm_probability=0.15):
    input_ids = examples["input_ids"]
    attention_mask = examples["attention_mask"]

    labels = []
    new_input_ids = []

    for seq in input_ids:
        seq = seq.copy()
        label = [-100] * len(seq)

        for i in range(len(seq)):
            if np.random.rand() < mlm_probability:
                label[i] = seq[i]
                seq[i] = tokenizer.mask_token_id

        new_input_ids.append(seq)
        labels.append(label)

    return {
        "input_ids": new_input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Adequando os Dados

Para treinar um modelo de linguagem de forma eficiente, precisamos **organizar os dados em lotes (batches)** e colocá-los em uma estrutura que o modelo consiga consumir durante o treinamento.  

**Dica:**
- Testem com diferentes tamanhos de `batch_size` e observem como isso afeta a memória e a velocidade.  
- Conferir os **shapes** dos tensores é sempre uma boa prática para evitar erros no treinamento.  

In [ ]:
# The 'text' and 'label' columns are already removed based on the error message.
# Apply manual masking
masked_tokenized_dataset = tokenized_dataset.map(
    lambda x: mask_tokens_manual(x, tokenizer, mlm_probability=0.15),
    batched=True
)

tf_train_dataset = masked_tokenized_dataset["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    shuffle=True,
    batch_size=8 # DICA: tente diferentes tamanhos de batch
)

tf_test_dataset = masked_tokenized_dataset["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    shuffle=False,
    batch_size=8 # DICA: tente diferentes tamanhos de batch
)

for batch in tf_train_dataset.take(1):
    for k, v in batch.items():
        print(f"{k}: {v.shape}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

input_ids: (8, 512)
attention_mask: (8, 512)
labels: (8, 512)


# Carregando dados de um modelo pré-treinado para continuar o pré-treinamento

Até aqui vocês aprenderam a preparar dados (tokenizar, mascarar e estruturar em batches).  
Agora, em vez de treinar um modelo do zero, vamos **aproveitar o conhecimento de um modelo já pré-treinado** e continuar o pré-treinamento com o nosso conjunto de dados (IMDB).  

Por que usar um modelo pré-treinado?
- Modelos como **DistilBERT** já foram treinados em enormes quantidades de texto.  
- Isso significa que eles **já entendem bastante da linguagem** (sintaxe, gramática, vocabulário, contexto).  
- Ao continuar o pré-treinamento com um novo corpus, você adapta o modelo para **captar melhor o estilo e o domínio** dos seus dados.  
- Lembre-se de experimentar diferentes quantidades de épocas e valores de learning-rate.  


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=masked_tokenized_dataset["train"],
    eval_dataset=masked_tokenized_dataset["test"],
)

trainer.train()

model.eval()

Step,Training Loss
500,1.132731
1000,0.757773


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBertForMaskedLM(
  (activation): GELUActivation()
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

# Testando as Predições

**Atenção use esses três exemplos para testar!!!**

In [ ]:
def predict_mask(text, model, tokenizer, top_k=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

    if len(mask_token_index) == 0:
        raise ValueError("O texto não contém token [MASK]")

    mask_logits = logits[0, mask_token_index, :]
    top_tokens = torch.topk(mask_logits, top_k, dim=1).indices[0].tolist()

    print(f"\nTexto: {text}")
    print("Previsões:")

    for token in top_tokens:
        word = tokenizer.decode([token])
        print("-", text.replace(tokenizer.mask_token, word))

In [ ]:
# Exemplo 1
predict_mask("The movie was really [MASK].", model, tokenizer)

# Exemplo 2
predict_mask("The food at the restaurant was absolutely [MASK].", model, tokenizer)

# Exemplo 3
predict_mask("The weather today is very [MASK].", model, tokenizer)


Texto: The movie was really [MASK].
Previsões:
- The movie was really good.
- The movie was really great.
- The movie was really bad.
- The movie was really funny.
- The movie was really entertaining.

Texto: The food at the restaurant was absolutely [MASK].
Previsões:
- The food at the restaurant was absolutely delicious.
- The food at the restaurant was absolutely fabulous.
- The food at the restaurant was absolutely amazing.
- The food at the restaurant was absolutely fantastic.
- The food at the restaurant was absolutely wonderful.

Texto: The weather today is very [MASK].
Previsões:
- The weather today is very bad.
- The weather today is very hot.
- The weather today is very cold.
- The weather today is very warm.
- The weather today is very good.


Busque alguma frase de exemplo da base de dados de teste e coloque a máscara em lugares variados do texto.

In [ ]:
words = subset_dataset['test'][1]['text'].split()
idx = np.random.randint(0, len(words))
words[idx] = tokenizer.mask_token
example_test_text = " ".join(words)

print(type(example_test_text))
predict_mask(example_test_text, model, tokenizer)

<class 'str'>

Texto: So far I disliked every single Jean Rollin movie I've seen, and that always bothered me because he's an acclaimed Euro-trash monument and extremely popular amongst many regular reviewers on this lovely website; people whose opinions I always value and usually concur with. Apparently everybody always appears to pinpoint some sort of gloomy and stylistic filming trademarks in his work that are completely lost on me. Rollin's movies are unimaginably boring, they all feature the same basic concept (lesbian vampires in various settings), the dialogs are incredibly absurd, the marvelous Gothic setting are always underused and the production values are cheaper than [MASK] price of a bus ticket. I had actually given up on Rollin's repertoire already (especially after enduring "The Iron Rose"), until I found out about "Night of the Hunted". Allegedly, this movie doesn't feature any lame lesbian vampires and stands as a bona fide horror movie with gruesome killings and maca

In [ ]:
text = "This movie was a [MASK] disappointment, but the acting was still good."
predict_mask(text, model, tokenizer)


Texto: This movie was a [MASK] disappointment, but the acting was still good.
Previsões:
- This movie was a huge disappointment, but the acting was still good.
- This movie was a great disappointment, but the acting was still good.
- This movie was a complete disappointment, but the acting was still good.
- This movie was a major disappointment, but the acting was still good.
- This movie was a big disappointment, but the acting was still good.


# Finetuning Downstream (Classificação)

Até agora o foco foi no **pré-treinamento com Masked Language Modeling (MLM)**, onde o objetivo era prever palavras mascaradas.  
Mas o poder do BERT/DistilBERT aparece de verdade quando usamos esse conhecimento adquirido em **tarefas downstream** (tarefas específicas), como classificação.  

**O que muda aqui?**
- No pré-treinamento, o modelo aprende sobre a **linguagem em geral**.  
- No fine-tuning, ajustamos o modelo para uma tarefa **específica**, por exemplo:  
  - **Classificação de sentimentos** (positivo/negativo em reviews do IMDB).  
  - **Classificação de tópicos**.  
  - **Detecção de spam**.  

**O que vocês precisam fazer aqui:**
- Utilizar um modelo pré-treinado (pode ser o seu ou não) (`DistilBERT`).  
- Adaptá-lo para classificação usando `TFDistilBertForSequenceClassification` (ou versão PyTorch).  
- Preparar os dados de entrada (frases e rótulos → `0` para negativo, `1` para positivo, por exemplo).  
- Treinar o modelo nos dados rotulados.
  
**Objetivo final:**  
Transformar um modelo genérico (pré-treinado em linguagem) em um modelo **especializado em classificar sentimentos no IMDB**.

**Dicas:**
- Ajustem **taxa de aprendizado** e **batch size**: esses hiperparâmetros têm forte impacto no desempenho.  
- Depois de treinar, calcule as métricas para verificar se o modelo capturou bem os padrões de classificação.
- Tente obter resultados melhores do que os do notebook exemplo da aula.  



In [ ]:
num_labels = len(set(tokenized_dataset['train']['label'])) # Assumindo 2 labels: 0 e 1
model_classification = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

training_args_classification = TrainingArguments(
    output_dir="./results_classification",
    eval_strategy="epoch",   # 👈 aqui está a correção
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

trainer_classification = Trainer(
    model=model_classification,
    args=training_args_classification,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

trainer_classification.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.321944,0.296597
2,0.218467,0.273641
3,0.129216,0.288687


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.2741793145338694, metrics={'train_runtime': 447.5088, 'train_samples_per_second': 13.408, 'train_steps_per_second': 0.838, 'total_flos': 794804391936000.0, 'train_loss': 0.2741793145338694, 'epoch': 3.0})

In [ ]:
metrics = trainer_classification.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch
0.129216,0.288687,3


{'eval_loss': 0.2886865735054016}


In [ ]:
def predict_sentiment(text, model, tokenizer):
    # Move model to CPU if it's on GPU (for easier CPU-only inference if needed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval() # Set model to evaluation mode

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Get the logits
    logits = outputs.logits

    # Apply softmax to get probabilities (optional, but good for understanding confidence)
    probabilities = torch.softmax(logits, dim=-1)

    # Get the predicted label (index with highest probability)
    predicted_class_id = torch.argmax(probabilities, dim=-1).item()

    # Map the class ID back to a human-readable label (assuming 0=negative, 1=positive for IMDB)
    if predicted_class_id == 0:
        return "Negative", probabilities[0][0].item()
    elif predicted_class_id == 1:
        return "Positive", probabilities[0][1].item()
    else:
        return "Unknown", None

# Example texts to test
example_texts = [
    "This movie was absolutely fantastic, a true masterpiece!",
    "The film was utterly boring and a complete waste of time.",
    "I had high hopes for this one, but it was just average.",
    "A compelling story with excellent performances, highly recommend!",
    "Never before have I been so disappointed by a movie, simply awful."
]

print("\n--- Sentiment Predictions ---")
for text in example_texts:
    sentiment, confidence = predict_sentiment(text, model_classification, tokenizer)
    print(f"\nText: '{text}'")
    print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.2f})")


--- Sentiment Predictions ---

Text: 'This movie was absolutely fantastic, a true masterpiece!'
Predicted Sentiment: Positive (Confidence: 0.98)

Text: 'The film was utterly boring and a complete waste of time.'
Predicted Sentiment: Negative (Confidence: 0.98)

Text: 'I had high hopes for this one, but it was just average.'
Predicted Sentiment: Negative (Confidence: 0.94)

Text: 'A compelling story with excellent performances, highly recommend!'
Predicted Sentiment: Positive (Confidence: 0.98)

Text: 'Never before have I been so disappointed by a movie, simply awful.'
Predicted Sentiment: Negative (Confidence: 0.99)
